In [1]:
import os, sys, json, time, random, warnings, gc
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix
)
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")

print("Python:", sys.executable)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError("CUDA is not available. Stop here and fix PyTorch/CUDA first.")

print("GPU:", torch.cuda.get_device_name(0))
print("CUDA:", torch.version.cuda)
print(
    "VRAM:",
    round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2),
    "GB"
)


Python: /home/adminbum/ntqai/.venv/bin/python
PyTorch: 2.14.1+cu130
CUDA available: True
GPU: NVIDIA GeForce RTX 4070 Ti SUPER
CUDA: 13.0
VRAM: 15.99 GB


/home/adminbum/ntqai/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
ROOT = Path("/mnt/d/Data/VNTM3/_stft30_v1")
SOURCE_REPORT_DIR = ROOT / "reports"
FEATURE_MANIFEST = SOURCE_REPORT_DIR / "stft_feature_manifest.csv"

RUN_ROOT = Path("/mnt/d/Data/VNTM3/_cnn2d_stft_multiseed_v1")
CACHE_DIR = RUN_ROOT / "cache_256x320"
RESULT_ROOT = RUN_ROOT / "results"
CHECKPOINT_ROOT = RUN_ROOT / "checkpoints"
FIGURE_ROOT = RUN_ROOT / "figures"

for p in [
    RUN_ROOT, CACHE_DIR, RESULT_ROOT,
    CHECKPOINT_ROOT, FIGURE_ROOT
]:
    p.mkdir(parents=True, exist_ok=True)

SEEDS = [42, 123, 2026]

INPUT_H = 256
INPUT_W = 320


BATCH_SIZE = 96
NUM_WORKERS = 8
PREFETCH_FACTOR = 2

EPOCHS = 50
LEARNING_RATE = 3e-4
WEIGHT_DECAY = 1e-4

PATIENCE = 10
MIN_DELTA = 1e-4


USE_TORCH_COMPILE = False

DEVICE = torch.device("cuda")


torch.backends.cudnn.benchmark = True
torch.backends.cudnn.deterministic = False


torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True


try:
    torch.set_float32_matmul_precision("high")
except Exception:
    pass

print("Run root:", RUN_ROOT)
print("Seeds:", SEEDS)
print("Batch size:", BATCH_SIZE)
print("Workers:", NUM_WORKERS)
print("Prefetch:", PREFETCH_FACTOR)
print("torch.compile:", USE_TORCH_COMPILE)


Run root: /mnt/d/Data/VNTM3/_cnn2d_stft_multiseed_v1
Seeds: [42, 123, 2026]
Batch size: 96
Workers: 8
Prefetch: 2
torch.compile: False


In [3]:
df = pd.read_csv(FEATURE_MANIFEST)

required = {"split", "label", "filename", "feature_path"}
missing = required - set(df.columns)

if missing:
    raise ValueError(f"Missing columns: {missing}")

classes = sorted(df["label"].unique())
class_to_idx = {c: i for i, c in enumerate(classes)}
idx_to_class = {i: c for c, i in class_to_idx.items()}
NUM_CLASSES = len(classes)

print("Files:", len(df))
display(
    df.groupby(["label","split"])
    .size()
    .unstack(fill_value=0)
)

assert NUM_CLASSES == 5
assert set(df["split"].unique()) == {"train","valid","test"}

print("Locked split loaded. No re-splitting.")


Files: 2495


split,test,train,valid
label,,,
cailuong,75,350,75
catru,75,347,74
chauvan,75,349,75
cheo,75,350,75
hatxam,75,350,75


Locked split loaded. No re-splitting.


In [4]:
def cache_path_for(row):
    split = row["split"]
    label = row["label"]
    stem = Path(row["filename"]).stem

    out_dir = CACHE_DIR / split / label
    out_dir.mkdir(parents=True, exist_ok=True)

    return out_dir / f"{stem}.npy"


cache_records = []
cache_errors = []

for _, row in tqdm(
    df.iterrows(),
    total=len(df),
    desc="Building/validating resize cache"
):
    out_path = cache_path_for(row)

    try:
        valid_cache = False

        if out_path.exists():
            arr = np.load(out_path, mmap_mode="r")
            valid_cache = (
                arr.shape == (INPUT_H, INPUT_W) and
                arr.dtype == np.float16
            )

        if not valid_cache:
            arr = np.load(row["feature_path"]).astype(np.float32)

            x = torch.from_numpy(arr)[None, None, :, :]

            x = F.interpolate(
                x,
                size=(INPUT_H, INPUT_W),
                mode="bilinear",
                align_corners=False
            )

            resized = (
                x[0, 0]
                .numpy()
                .astype(np.float16)
            )

            np.save(out_path, resized)

        cache_records.append({
            "split": row["split"],
            "label": row["label"],
            "filename": row["filename"],
            "original_feature_path": row["feature_path"],
            "cache_path": str(out_path)
        })

    except Exception as e:
        cache_errors.append({
            "filename": row["filename"],
            "error": repr(e)
        })

cache_df = pd.DataFrame(cache_records)
cache_errors_df = pd.DataFrame(cache_errors)

print("Cache OK:", len(cache_df))
print("Cache errors:", len(cache_errors_df))

assert len(cache_errors_df) == 0
assert len(cache_df) == len(df)

cache_df.to_csv(
    RUN_ROOT / "cache_manifest.csv",
    index=False,
    encoding="utf-8-sig"
)


Building/validating resize cache: 100%|██████████| 2495/2495 [01:27<00:00, 28.67it/s]

Cache OK: 2495
Cache errors: 0


In [5]:
sample_cache = np.load(cache_df.iloc[0]["cache_path"])

print("Cache sample shape:", sample_cache.shape)
print("dtype:", sample_cache.dtype)
print(
    "Approx raw cache payload:",
    round(
        len(cache_df) * INPUT_H * INPUT_W * 2 / 1024**3,
        3
    ),
    "GiB"
)

assert sample_cache.shape == (INPUT_H, INPUT_W)
assert sample_cache.dtype == np.float16


Cache sample shape: (256, 320)
dtype: float16
Approx raw cache payload: 0.381 GiB


In [6]:
class CachedSTFTDataset(Dataset):
    def __init__(self, dataframe, class_to_idx, train=False):
        self.df = dataframe.reset_index(drop=True)
        self.class_to_idx = class_to_idx
        self.train = train

    def __len__(self):
        return len(self.df)

    def _augment(self, x):
        

        if random.random() < 0.50:
            max_shift = max(1, x.shape[-1] // 20)
            shift = random.randint(-max_shift, max_shift)
            x = torch.roll(x, shifts=shift, dims=-1)

        if random.random() < 0.50:
            max_width = max(1, x.shape[-2] // 12)
            width = random.randint(1, max_width)
            start = random.randint(0, x.shape[-2] - width)
            x[:, start:start+width, :] = 0.0

        if random.random() < 0.50:
            max_width = max(1, x.shape[-1] // 10)
            width = random.randint(1, max_width)
            start = random.randint(0, x.shape[-1] - width)
            x[:, :, start:start+width] = 0.0

        if random.random() < 0.40:
            scale = random.uniform(0.90, 1.10)
            x = torch.clamp(x * scale, 0.0, 1.0)

        return x

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        arr = np.load(row["cache_path"]).astype(
            np.float32,
            copy=False
        )

        x = torch.from_numpy(arr).unsqueeze(0)

        if self.train:
            x = self._augment(x)

        y = self.class_to_idx[row["label"]]

        return x, y, idx


In [7]:
class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, dropout=0.0):
        super().__init__()

        layers = [
            nn.Conv2d(
                in_ch, out_ch, 3,
                padding=1, bias=False
            ),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),

            nn.Conv2d(
                out_ch, out_ch, 3,
                padding=1, bias=False
            ),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),

            nn.MaxPool2d(2)
        ]

        if dropout > 0:
            layers.append(nn.Dropout2d(dropout))

        self.block = nn.Sequential(*layers)

    def forward(self, x):
        return self.block(x)


class CNN2D_STFT(nn.Module):
    def __init__(self, n_classes=5):
        super().__init__()

        self.features = nn.Sequential(
            ConvBlock(1, 32, 0.05),
            ConvBlock(32, 64, 0.10),
            ConvBlock(64, 128, 0.15),
            ConvBlock(128, 192, 0.20)
        )

        self.pool = nn.AdaptiveAvgPool2d((1,1))

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(192, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(0.40),
            nn.Linear(128, n_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.pool(x)
        return self.classifier(x)


In [8]:
def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    
    torch.backends.cudnn.benchmark = True
    torch.backends.cudnn.deterministic = False


def compute_metrics(y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)

    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true,
        y_pred,
        average="macro",
        zero_division=0
    )

    return {
        "accuracy": float(acc),
        "macro_precision": float(precision),
        "macro_recall": float(recall),
        "macro_f1": float(f1)
    }


In [9]:
def worker_init_fn(worker_id):
    
    worker_seed = torch.initial_seed() % (2**32)
    np.random.seed(worker_seed)
    random.seed(worker_seed)


def make_loaders(seed):
    train_df = cache_df[
        cache_df["split"] == "train"
    ].copy()

    valid_df = cache_df[
        cache_df["split"] == "valid"
    ].copy()

    test_df = cache_df[
        cache_df["split"] == "test"
    ].copy()

    train_ds = CachedSTFTDataset(
        train_df, class_to_idx, train=True
    )
    valid_ds = CachedSTFTDataset(
        valid_df, class_to_idx, train=False
    )
    test_ds = CachedSTFTDataset(
        test_df, class_to_idx, train=False
    )

    generator = torch.Generator()
    generator.manual_seed(seed)

    common = dict(
        batch_size=BATCH_SIZE,
        num_workers=NUM_WORKERS,
        pin_memory=True,
        persistent_workers=(NUM_WORKERS > 0),
        worker_init_fn=worker_init_fn
    )

    if NUM_WORKERS > 0:
        common["prefetch_factor"] = PREFETCH_FACTOR

    train_loader = DataLoader(
        train_ds,
        shuffle=True,
        generator=generator,
        **common
    )

    valid_loader = DataLoader(
        valid_ds,
        shuffle=False,
        **common
    )

    test_loader = DataLoader(
        test_ds,
        shuffle=False,
        **common
    )

    return (
        train_df, valid_df, test_df,
        train_loader, valid_loader, test_loader
    )


In [10]:
def run_epoch(
    model,
    loader,
    criterion,
    scaler,
    optimizer=None
):
    is_train = optimizer is not None

    model.train() if is_train else model.eval()

    total_loss = 0.0
    all_true = []
    all_pred = []

    for x, y, _ in loader:
        x = x.to(
            DEVICE,
            non_blocking=True,
            memory_format=torch.channels_last
        )

        y = y.to(
            DEVICE,
            non_blocking=True
        )

        if is_train:
            optimizer.zero_grad(set_to_none=True)

        with torch.set_grad_enabled(is_train):
            with torch.autocast(
                device_type="cuda",
                dtype=torch.float16,
                enabled=True
            ):
                logits = model(x)
                loss = criterion(logits, y)

            if is_train:
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()

        total_loss += loss.item() * x.size(0)

        pred = logits.argmax(dim=1)

        all_true.extend(
            y.detach().cpu().numpy().tolist()
        )
        all_pred.extend(
            pred.detach().cpu().numpy().tolist()
        )

    metrics = compute_metrics(
        all_true,
        all_pred
    )

    metrics["loss"] = (
        total_loss / len(loader.dataset)
    )

    return metrics


In [11]:
@torch.inference_mode()
def predict_full(model, loader):
    model.eval()

    all_true = []
    all_pred = []
    all_prob = []
    all_idx = []

    for x, y, idx in loader:
        x = x.to(
            DEVICE,
            non_blocking=True,
            memory_format=torch.channels_last
        )

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16
        ):
            logits = model(x)

        prob = torch.softmax(
            logits.float(),
            dim=1
        )
        pred = prob.argmax(dim=1)

        all_true.extend(y.numpy().tolist())
        all_pred.extend(pred.cpu().numpy().tolist())
        all_prob.append(prob.cpu().numpy())
        all_idx.extend(idx.numpy().tolist())

    return (
        np.array(all_true),
        np.array(all_pred),
        np.concatenate(all_prob, axis=0),
        np.array(all_idx)
    )


In [12]:
def train_one_seed(seed):
    print("\n" + "="*78)
    print(f"SEED {seed}")
    print("="*78)

    seed_everything(seed)

    (
        train_df, valid_df, test_df,
        train_loader, valid_loader, test_loader
    ) = make_loaders(seed)

    seed_result_dir = RESULT_ROOT / f"seed_{seed}"
    seed_ckpt_dir = CHECKPOINT_ROOT / f"seed_{seed}"
    seed_fig_dir = FIGURE_ROOT / f"seed_{seed}"

    for p in [
        seed_result_dir,
        seed_ckpt_dir,
        seed_fig_dir
    ]:
        p.mkdir(parents=True, exist_ok=True)

    model = CNN2D_STFT(NUM_CLASSES).to(
        DEVICE,
        memory_format=torch.channels_last
    )

    n_params = sum(
        p.numel()
        for p in model.parameters()
    )

    if USE_TORCH_COMPILE:
        print("Compiling model...")
        model = torch.compile(
            model,
            mode="reduce-overhead"
        )

    criterion = nn.CrossEntropyLoss()

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY
    )

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="max",
        factor=0.5,
        patience=3,
        min_lr=1e-6
    )

    scaler = torch.amp.GradScaler(
        "cuda",
        enabled=True
    )

    best_f1 = -1.0
    best_epoch = 0
    no_improve = 0
    history = []

    best_path = (
        seed_ckpt_dir /
        "best_cnn2d_stft.pt"
    )

    start = time.time()

    for epoch in range(1, EPOCHS + 1):
        train_m = run_epoch(
            model,
            train_loader,
            criterion,
            scaler,
            optimizer
        )

        valid_m = run_epoch(
            model,
            valid_loader,
            criterion,
            scaler,
            optimizer=None
        )

        scheduler.step(
            valid_m["macro_f1"]
        )

        lr_now = optimizer.param_groups[0]["lr"]

        history.append({
            "epoch": epoch,
            "lr": lr_now,
            **{
                f"train_{k}": v
                for k, v in train_m.items()
            },
            **{
                f"valid_{k}": v
                for k, v in valid_m.items()
            }
        })

        print(
            f"Seed {seed} | "
            f"Epoch {epoch:02d}/{EPOCHS} | "
            f"train acc={train_m['accuracy']:.4f} "
            f"F1={train_m['macro_f1']:.4f} | "
            f"valid acc={valid_m['accuracy']:.4f} "
            f"F1={valid_m['macro_f1']:.4f} | "
            f"lr={lr_now:.2e}"
        )

        current_f1 = valid_m["macro_f1"]

        if current_f1 > best_f1 + MIN_DELTA:
            best_f1 = current_f1
            best_epoch = epoch
            no_improve = 0

            
            state_model = (
                model._orig_mod
                if hasattr(model, "_orig_mod")
                else model
            )

            torch.save({
                "seed": seed,
                "epoch": epoch,
                "model_state_dict": state_model.state_dict(),
                "valid_metrics": valid_m,
                "classes": classes,
                "input_hw": [INPUT_H, INPUT_W]
            }, best_path)

        else:
            no_improve += 1

        if no_improve >= PATIENCE:
            print(
                f"Early stopping. "
                f"Best epoch={best_epoch}"
            )
            break

    elapsed = time.time() - start

    history_df = pd.DataFrame(history)
    history_df.to_csv(
        seed_result_dir /
        "training_history.csv",
        index=False
    )

    
    eval_model = CNN2D_STFT(
        NUM_CLASSES
    ).to(
        DEVICE,
        memory_format=torch.channels_last
    )

    checkpoint = torch.load(
        best_path,
        map_location=DEVICE
    )

    eval_model.load_state_dict(
        checkpoint["model_state_dict"]
    )

    y_true, y_pred, y_prob, ds_idx = predict_full(
        eval_model,
        test_loader
    )

    test_metrics = compute_metrics(
        y_true,
        y_pred
    )

    report = classification_report(
        y_true,
        y_pred,
        target_names=classes,
        output_dict=True,
        zero_division=0
    )

    report_df = pd.DataFrame(report).T
    report_df.to_csv(
        seed_result_dir /
        "classification_report.csv",
        encoding="utf-8-sig"
    )

    cm = confusion_matrix(
        y_true,
        y_pred,
        labels=list(range(NUM_CLASSES))
    )

    cm_df = pd.DataFrame(
        cm,
        index=classes,
        columns=classes
    )

    cm_df.to_csv(
        seed_result_dir /
        "confusion_matrix.csv",
        encoding="utf-8-sig"
    )

    pred_rows = []

    for n, idx in enumerate(ds_idx):
        row = test_df.iloc[int(idx)]

        item = {
            "filename": row["filename"],
            "label": row["label"],
            "predicted_label": idx_to_class[
                int(y_pred[n])
            ],
            "correct": bool(
                y_true[n] == y_pred[n]
            ),
            "confidence": float(
                np.max(y_prob[n])
            )
        }

        for cid, cname in idx_to_class.items():
            item[f"prob_{cname}"] = float(
                y_prob[n, cid]
            )

        pred_rows.append(item)

    pd.DataFrame(pred_rows).to_csv(
        seed_result_dir /
        "test_predictions.csv",
        index=False,
        encoding="utf-8-sig"
    )

    
    plt.figure(figsize=(9,5))
    plt.plot(
        history_df["epoch"],
        history_df["train_macro_f1"],
        label="train macro-F1"
    )
    plt.plot(
        history_df["epoch"],
        history_df["valid_macro_f1"],
        label="valid macro-F1"
    )
    plt.xlabel("Epoch")
    plt.ylabel("Macro-F1")
    plt.title(f"Seed {seed}")
    plt.legend()
    plt.tight_layout()
    plt.savefig(
        seed_fig_dir /
        "macro_f1_curve.png",
        dpi=160
    )
    plt.close()

    summary = {
        "seed": int(seed),
        "best_epoch": int(best_epoch),
        "best_valid_macro_f1": float(best_f1),
        "test_accuracy": float(
            test_metrics["accuracy"]
        ),
        "test_macro_precision": float(
            test_metrics["macro_precision"]
        ),
        "test_macro_recall": float(
            test_metrics["macro_recall"]
        ),
        "test_macro_f1": float(
            test_metrics["macro_f1"]
        ),
        "training_minutes": float(
            elapsed / 60
        ),
        "batch_size": BATCH_SIZE,
        "num_workers": NUM_WORKERS,
        "parameters": int(n_params)
    }

    with open(
        seed_result_dir /
        "summary.json",
        "w",
        encoding="utf-8"
    ) as f:
        json.dump(
            summary,
            f,
            ensure_ascii=False,
            indent=2
        )

    print("\nTEST")
    print(
        f"Accuracy={summary['test_accuracy']:.4f} | "
        f"Macro-F1={summary['test_macro_f1']:.4f} | "
        f"Time={summary['training_minutes']:.1f} min"
    )

    
    del model, eval_model
    del train_loader, valid_loader, test_loader
    gc.collect()
    torch.cuda.empty_cache()

    return summary, report_df


In [13]:
all_summaries = []
all_reports = {}

overall_start = time.time()

for seed in SEEDS:
    summary, report_df = train_one_seed(seed)

    all_summaries.append(summary)
    all_reports[seed] = report_df

total_minutes = (
    time.time() - overall_start
) / 60

print(
    f"\nAll seeds completed in "
    f"{total_minutes:.1f} minutes."
)



SEED 42
Seed 42 | Epoch 01/50 | train acc=0.4622 F1=0.4605 | valid acc=0.2005 F1=0.0668 | lr=3.00e-04
Seed 42 | Epoch 02/50 | train acc=0.5899 F1=0.5690 | valid acc=0.3636 F1=0.2134 | lr=3.00e-04
Seed 42 | Epoch 03/50 | train acc=0.6581 F1=0.6343 | valid acc=0.4733 F1=0.3881 | lr=3.00e-04
Seed 42 | Epoch 04/50 | train acc=0.6907 F1=0.6851 | valid acc=0.6738 F1=0.6516 | lr=3.00e-04
Seed 42 | Epoch 05/50 | train acc=0.7108 F1=0.7042 | valid acc=0.6497 F1=0.6407 | lr=3.00e-04
Seed 42 | Epoch 06/50 | train acc=0.7314 F1=0.7293 | valid acc=0.7995 F1=0.8000 | lr=3.00e-04
Seed 42 | Epoch 07/50 | train acc=0.7640 F1=0.7613 | valid acc=0.8262 F1=0.8233 | lr=3.00e-04
Seed 42 | Epoch 08/50 | train acc=0.7680 F1=0.7670 | valid acc=0.7968 F1=0.7991 | lr=3.00e-04
Seed 42 | Epoch 09/50 | train acc=0.7927 F1=0.7924 | valid acc=0.8102 F1=0.8039 | lr=3.00e-04
Seed 42 | Epoch 10/50 | train acc=0.7961 F1=0.7942 | valid acc=0.8503 F1=0.8451 | lr=3.00e-04
Seed 42 | Epoch 11/50 | train acc=0.8121 F1=0.8108 

In [14]:
summary_df = pd.DataFrame(
    all_summaries
)

summary_df.to_csv(
    RESULT_ROOT /
    "multiseed_per_seed.csv",
    index=False,
    encoding="utf-8-sig"
)

display(summary_df)

metric_cols = [
    "test_accuracy",
    "test_macro_precision",
    "test_macro_recall",
    "test_macro_f1"
]

aggregate_rows = []

for metric in metric_cols:
    values = summary_df[metric].to_numpy()

    aggregate_rows.append({
        "metric": metric,
        "mean": float(values.mean()),
        "std": float(values.std(ddof=1)),
        "min": float(values.min()),
        "max": float(values.max())
    })

aggregate_df = pd.DataFrame(
    aggregate_rows
)

aggregate_df.to_csv(
    RESULT_ROOT /
    "multiseed_mean_std.csv",
    index=False,
    encoding="utf-8-sig"
)

display(aggregate_df)


,seed,best_epoch,best_valid_macro_f1,test_accuracy,test_macro_precision,test_macro_recall,test_macro_f1,training_minutes,batch_size,num_workers,parameters
0,42,29,0.978540,0.984000,0.983997,0.984000,0.983946,1.812277,96,8,865957
1,123,44,0.975814,0.976000,0.976918,0.976000,0.975945,2.157145,96,8,865957
2,2026,46,0.957194,0.957333,0.959598,0.957333,0.956774,2.052378,96,8,865957


,metric,mean,std,min,max
0,test_accuracy,0.972444,0.013684,0.957333,0.984000
1,test_macro_precision,0.973504,0.012552,0.959598,0.983997
2,test_macro_recall,0.972444,0.013684,0.957333,0.984000
3,test_macro_f1,0.972221,0.013963,0.956774,0.983946


In [15]:
per_class_rows = []

for class_name in classes:
    for metric in [
        "precision",
        "recall",
        "f1-score"
    ]:
        values = np.array([
            all_reports[seed]
            .loc[class_name, metric]
            for seed in SEEDS
        ], dtype=float)

        per_class_rows.append({
            "class": class_name,
            "metric": metric,
            "mean": float(values.mean()),
            "std": float(values.std(ddof=1)),
            "min": float(values.min()),
            "max": float(values.max())
        })

per_class_df = pd.DataFrame(
    per_class_rows
)

per_class_df.to_csv(
    RESULT_ROOT /
    "multiseed_per_class.csv",
    index=False,
    encoding="utf-8-sig"
)

display(per_class_df)


,class,metric,mean,std,min,max
0,cailuong,precision,0.995434,0.007909,0.986301,1.000000
1,cailuong,recall,0.911111,0.053886,0.853333,0.960000
2,cailuong,f1-score,0.950723,0.026875,0.920863,0.972973
3,catru,precision,0.986842,0.013158,0.973684,1.000000
4,catru,recall,0.991111,0.007698,0.986667,1.000000
5,catru,f1-score,0.988933,0.007621,0.980132,0.993377
6,chauvan,precision,0.937886,0.034874,0.903614,0.973333
7,chauvan,recall,0.986667,0.013333,0.973333,1.000000
8,chauvan,f1-score,0.961246,0.011984,0.949367,0.973333
9,cheo,precision,0.982570,0.007399,0.974026,0.986842


In [16]:
acc = summary_df[
    "test_accuracy"
].to_numpy()

f1 = summary_df[
    "test_macro_f1"
].to_numpy()

final_summary = {
    "experiment": "CNN2D_STFT_30s_multiseed_v1",
    "seeds": SEEDS,
    "fixed_split": True,
    "batch_size": BATCH_SIZE,
    "num_workers": NUM_WORKERS,
    "input_shape": [1, INPUT_H, INPUT_W],
    "amp_fp16": True,
    "channels_last": True,
    "resize_cache": True,
    "torch_compile": USE_TORCH_COMPILE,
    "test_accuracy_mean": float(acc.mean()),
    "test_accuracy_std": float(acc.std(ddof=1)),
    "test_macro_f1_mean": float(f1.mean()),
    "test_macro_f1_std": float(f1.std(ddof=1)),
    "total_training_minutes": float(total_minutes)
}

with open(
    RESULT_ROOT /
    "multiseed_summary.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        final_summary,
        f,
        ensure_ascii=False,
        indent=2
    )

print(json.dumps(
    final_summary,
    ensure_ascii=False,
    indent=2
))


{
  "experiment": "CNN2D_STFT_30s_multiseed_v1",
  "seeds": [
    42,
    123,
    2026
  ],
  "fixed_split": true,
  "batch_size": 96,
  "num_workers": 8,
  "input_shape": [
    1,
    256,
    320
  ],
  "amp_fp16": true,
  "channels_last": true,
  "resize_cache": true,
  "torch_compile": false,
  "test_accuracy_mean": 0.9724444444444446,
  "test_accuracy_std": 0.013684270505191663,
  "test_macro_f1_mean": 0.9722214759351866,
  "test_macro_f1_std": 0.013962992701270922,
  "total_training_minutes": 6.086049874623616
}


In [17]:
print("="*78)
print("CNN2D-STFT MULTI-SEED FINAL")
print("="*78)

for row in all_summaries:
    print(
        f"Seed {row['seed']:4d} | "
        f"Acc={row['test_accuracy']:.4f} | "
        f"Macro-F1={row['test_macro_f1']:.4f} | "
        f"Best epoch={row['best_epoch']:02d} | "
        f"{row['training_minutes']:.1f} min"
    )

print()
print(
    "TEST ACCURACY = "
    f"{acc.mean():.4f} ± "
    f"{acc.std(ddof=1):.4f}"
)
print(
    "TEST MACRO-F1 = "
    f"{f1.mean():.4f} ± "
    f"{f1.std(ddof=1):.4f}"
)
print()
print("Results:", RESULT_ROOT)


CNN2D-STFT MULTI-SEED FINAL
Seed   42 | Acc=0.9840 | Macro-F1=0.9839 | Best epoch=29 | 1.8 min
Seed  123 | Acc=0.9760 | Macro-F1=0.9759 | Best epoch=44 | 2.2 min
Seed 2026 | Acc=0.9573 | Macro-F1=0.9568 | Best epoch=46 | 2.1 min

TEST ACCURACY = 0.9724 ± 0.0137
TEST MACRO-F1 = 0.9722 ± 0.0140

Results: /mnt/d/Data/VNTM3/_cnn2d_stft_multiseed_v1/results
